# Advanced NLP Assignment — Group 6
## Scaling Laws, Supervised Fine-Tuning, and Parameter-Efficient Adaptation

**Course:** Advanced NLP  
**Authors:** Group 6  
**Base codebase:** [nanoGPT](https://github.com/karpathy/nanoGPT) (Karpathy, 2023)

This notebook contains all code for Parts 1, 2, and 3 of the assignment, organized with markdown cells explaining each step.

---
## Setup

In [ ]:
import os
import json
import math
import pickle
import random
import re
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 150, 'font.size': 11,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
})

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

---
# Part 1: Scaling Laws

We train a grid of models varying in size (XS, S, M, L) and dataset size (10%, 25%, 50%, 100%) on the Tiny Shakespeare character-level dataset, measuring final validation loss for each combination.

## 1.1 Data Preparation — Dataset Subsets

In [ ]:
# Create train subsets at 10%, 25%, 50%, 100% of training data
data = np.memmap('data/shakespeare_char/train.bin', dtype=np.uint16, mode='r')
print(f'Full training data: {len(data):,} tokens')

for frac in [0.10, 0.25, 0.50, 1.00]:
    subset = data[:int(len(data) * frac)]
    out_path = f'data/shakespeare_char/train_{int(frac*100)}.bin'
    subset.tofile(out_path)
    print(f'  train_{int(frac*100)}.bin: {len(subset):,} tokens')

## 1.2 Model Configurations

| Label | n_layer | n_head | n_embd | Approx. params |
|-------|---------|--------|--------|----------------|
| XS    | 2       | 2      | 64     | ~0.1M          |
| S     | 4       | 4      | 128    | ~0.8M          |
| M     | 6       | 6      | 384    | ~10M           |
| L     | 8       | 8      | 512    | ~25M           |

In [ ]:
# Training commands for the 16-run grid
# Each command was run on Lightning AI (T4 GPU)

model_configs = {
    'XS': {'n_layer': 2, 'n_head': 2, 'n_embd': 64},
    'S':  {'n_layer': 4, 'n_head': 4, 'n_embd': 128},
    'M':  {'n_layer': 6, 'n_head': 6, 'n_embd': 384},
    'L':  {'n_layer': 8, 'n_head': 8, 'n_embd': 512},
}

dataset_fracs = [10, 25, 50, 100]

# Example training command (M_100 baseline):
cmd = """
python train.py \
  --n_layer=6 --n_head=6 --n_embd=384 --bias=False \
  --dataset=shakespeare_char --train_file=train_100.bin \
  --block_size=256 --batch_size=32 --max_iters=610 \
  --lr_decay_iters=610 --learning_rate=1e-3 --min_lr=1e-4 \
  --beta2=0.99 --warmup_iters=100 --weight_decay=0.1 \
  --grad_clip=1.0 --dropout=0.0 --device=cuda --compile=False \
  --out_dir=out/scaling/M_100 --always_save_checkpoint=True
"""
print("Example training command (M_100):")
print(cmd)

## 1.3 Results — Loss Grid

In [ ]:
# Load scaling results
with open('scaling_results.json') as f:
    scaling_results = json.load(f)

# Display loss grid
sizes = ['XS', 'S', 'M', 'L']
fracs = [10, 25, 50, 100]

print(f"{'':>6} | {'10%':>8} {'25%':>8} {'50%':>8} {'100%':>8}")
print('-' * 45)
for size in sizes:
    row = f"{size:>6} |"
    for frac in fracs:
        key = f"{size}_{frac}"
        val = scaling_results.get(key, {}).get('val_loss', float('nan'))
        row += f" {val:>8.4f}"
    print(row)

## 1.4 Scaling Plots

In [ ]:
# Run the scaling analysis script
%run analyze_scaling.py

## 1.5 Discussion

**Q1.** Do you observe a clear compute-optimal trade-off between model size and data size?

*(See report for full answers to Q1–Q3)*

---
# Part 2: Supervised Fine-Tuning (SFT)

We fine-tune the M_100 pretrained checkpoint on two tasks:
- **Task A:** Speaker identification — given a Shakespeare line, predict the speaker
- **Task B:** Verse vs. Prose classification

## 2.1 Data Preparation — Task A (Speaker Identification)

In [ ]:
# Task A format: [SPEAKER] <dialogue> [ANSWER] CHARACTER [END]

DATA_DIR_A = os.path.join('data', 'task_a')

with open(os.path.join(DATA_DIR_A, 'train.txt'), encoding='utf-8') as f:
    train_lines_a = [l.strip() for l in f if l.strip()]
with open(os.path.join(DATA_DIR_A, 'test.txt'), encoding='utf-8') as f:
    test_lines_a = [l.strip() for l in f if l.strip()]

print(f'Task A — Train: {len(train_lines_a)} examples, Test: {len(test_lines_a)} examples')
print(f'\nSample example:')
print(train_lines_a[0])

In [ ]:
# Encode Task A data to binary
%run prepare_task_a.py

## 2.2 Data Preparation — Task B (Verse vs. Prose)

In [ ]:
# Task B format: [CLASSIFY] <passage> [ANSWER] VERSE/PROSE [END]
# Heuristic: blocks are sorted by average line length.
# Bottom half (shorter lines) = VERSE, top half (longer lines) = PROSE.

%run prepare_task_b.py

## 2.3 Fine-Tuning

In [ ]:
# Prepare M_100 checkpoint for fine-tuning (add missing keys)
import torch

for task in ['task_a', 'task_b', 'multitask']:
    os.makedirs(f'out/{task}', exist_ok=True)
    src = 'out/scaling/M_100/ckpt_small.pt'
    dst = f'out/{task}/ckpt.pt'
    if not os.path.exists(dst):
        ckpt = torch.load(src, map_location='cpu')
        ckpt['iter_num'] = 0
        ckpt['best_val_loss'] = 2.04
        ckpt['optimizer'] = {}
        torch.save(ckpt, dst)
        print(f'Prepared {dst}')
    else:
        print(f'Already exists: {dst}')

In [ ]:
# Fine-tune on Task A (single-task)
# Hyperparams: lr=5e-5, max_iters=2000, batch_size=32, dropout=0.1
!python train.py config/finetune_task_a.py 2>&1 | tee logs/task_a.log

In [ ]:
# Fine-tune on Task B (single-task)
!python train.py config/finetune_task_b.py 2>&1 | tee logs/task_b.log

In [ ]:
# Multi-task fine-tuning (Task A + Task B interleaved)
!python train.py config/finetune_multitask.py 2>&1 | tee logs/multitask.log

## 2.4 Training Curves

In [ ]:
%run plot_sft.py
from IPython.display import Image
Image('sft_training_curves.png')

## 2.5 Evaluation

In [ ]:
# Task A accuracy
!python eval.py --task=a --checkpoint=out/task_a/ckpt.pt --data=data/task_a/test.txt --device=cuda

In [ ]:
# Task B accuracy
!python eval.py --task=b --checkpoint=out/task_b/ckpt.pt --data=data/task_b/test.txt --device=cuda

In [ ]:
# Multitask accuracy on both tasks
!python eval.py --task=a --checkpoint=out/multitask/ckpt.pt --data=data/task_a/test.txt --device=cuda
!python eval.py --task=b --checkpoint=out/multitask/ckpt.pt --data=data/task_b/test.txt --device=cuda

In [ ]:
# Catastrophic forgetting — Shakespeare val loss for each fine-tuned model
!python eval.py --task=forgetting --checkpoint=out/task_a/ckpt.pt --device=cuda
!python eval.py --task=forgetting --checkpoint=out/task_b/ckpt.pt --device=cuda
!python eval.py --task=forgetting --checkpoint=out/multitask/ckpt.pt --device=cuda

## 2.6 Results Table

| Setup | Task A Acc. | Task B Acc. | Shakespeare Val Loss |
|-------|-------------|-------------|---------------------|
| Random baseline | 10.0% | 50.0% | — |
| Pretrained (no SFT) | — | — | 2.04 |
| Single-task A | 19.0% | — | 5.75 |
| Single-task B | — | 68.89% | 6.25 |
| Multi-task A+B | 16.0% | 63.33% | 8.44 |

## 2.7 Generated Samples (Experiment 3 — Catastrophic Forgetting)

In [ ]:
# Pretrained baseline
!python sample.py --out_dir=out/scaling/M_100 --device=cuda --num_samples=3 --max_new_tokens=200 --start="ROMEO:"

In [ ]:
# Task A fine-tuned
!python sample.py --out_dir=out/task_a --device=cuda --num_samples=3 --max_new_tokens=200 --start="ROMEO:"

In [ ]:
# Task B fine-tuned
!python sample.py --out_dir=out/task_b --device=cuda --num_samples=3 --max_new_tokens=200 --start="ROMEO:"

---
# Part 3: Low-Rank Adaptation (LoRA)

We implement LoRA from scratch in `model_lora.py`, injecting trainable rank-decomposition matrices into the Q and V projections of each attention layer while freezing all base weights.

## 3.1 LoRA Implementation

Key design choices:
- `LoRALinear` wraps any `nn.Linear`: `output = W·x + (x·Aᵀ)·Bᵀ`
- A initialized with small Gaussian (0.01), B initialized with zeros → initial LoRA contribution = 0
- Applied to Q and V projections only (as per Hu et al., 2022)
- `inject_lora()` freezes all base params, then wraps Q/V with LoRALinear

In [ ]:
# Sanity check: verify LoRA implementation correctness
# Test 1: only LoRA params have requires_grad=True
# Test 2: output identical to pretrained before any training (B=0)
!python model_lora.py out/scaling/M_100/ckpt_small.pt 4

## 3.2 Experiments

In [ ]:
# Run all LoRA experiments:
# Experiment 4: LoRA rank=4 single-task
# Experiment 5: Rank ablation r ∈ {1, 2, 4, 8, 16}
# Experiment 6: Full fine-tuning vs LoRA comparison
!python train_lora.py --experiment=all --device=cuda

## 3.3 Results

In [ ]:
# Load and display LoRA results
with open('lora_results.json') as f:
    lora_results = json.load(f)

pretrained_val = lora_results.get('pretrained_val_loss', 2.04)
ranks = [1, 2, 4, 8, 16]

print(f"{'Method':<20} {'Rank':>6} {'Trainable Params':>18} {'Val Loss':>10} {'Forgetting':>12} {'Time (min)':>12}")
print('-' * 82)

ft = lora_results.get('full_finetune')
if ft:
    forgetting = ft['val_loss'] - pretrained_val
    print(f"{'Full Fine-Tuning':<20} {'—':>6} {ft['trainable_params']:>18,} {ft['val_loss']:>10.4f} {forgetting:>+12.4f} {ft['elapsed_s']/60:>12.1f}")

for rank in ranks:
    v = lora_results.get(f'lora_rank_{rank}')
    if v:
        forgetting = v['val_loss'] - pretrained_val
        print(f"{'LoRA':<20} {rank:>6} {v['trainable_params']:>18,} {v['val_loss']:>10.4f} {forgetting:>+12.4f} {v['elapsed_s']/60:>12.1f}")

## 3.4 Rank Ablation Plot

In [ ]:
from IPython.display import Image
Image('lora_1_rank_ablation.png')

## 3.5 Discussion

**Q7.** At what rank does LoRA match or approach full fine-tuning accuracy?

LoRA at rank 4 achieves val loss 2.045, outperforming full fine-tuning (2.205) with only 0.34% of the trainable parameters. Even rank 1 nearly matches full FT, suggesting very low intrinsic dimensionality of the adaptation.

**Q8.** How does LoRA compare to full fine-tuning in terms of catastrophic forgetting?

LoRA shows significantly less forgetting (delta +0.005 at rank 4) vs full FT (delta +0.165), because frozen base weights preserve the pretrained representations structurally.

**Q9.** If deploying for 10 tasks simultaneously, what practical advantages does LoRA offer?

One shared frozen base model + 10 small adapters (~36K params each at rank 4) vs 10 full copies of 10.7M params. Task switching = swap LoRA weights only. Training is faster and more memory-efficient.